# TP  Pronostic Coupe du Monde FIFA 2026   **TensorFlow / Keras**

**M2 DMIA  Deep Learning  Projet 1 (réseau dense / MLP)**

Objectif : prédire **l'issue d'un match** de football international  victoire de l'équipe **domicile**, **match nul**, ou victoire de l'équipe **extérieur**  à partir de statistiques d'équipes (forme récente, buts marqués/encaissés, avantage du terrain…).

### Ce que tu vas apprendre
- Ce qu'est concrètement **un neurone** : `entrées × poids + biais → activation`. Sur des données **tabulaires** (un tableau de chiffres), on le voit littéralement.
- La **classification multi-classes** avec la couche **softmax** (3 probabilités qui somment à 1).
- Comment **observer le sur-apprentissage** (les courbes train vs validation qui divergent) et le **corriger** avec le **dropout**.
- Bonus : transformer les probabilités du modèle en une **mini-simulation de tournoi**.

On suit le **pipeline DL en 7 étapes** vu en cours :
1. Charger les données → 2. Construire les features → 3. Prétraiter → 4. Définir le modèle → 5. Entraîner → 6. Évaluer → 7. Analyser.

>  **Note Colab** : ce notebook tourne aussi sur **Google Colab** (Exécution → Modifier le type d'exécution → GPU est inutile ici, le CPU suffit). Il fonctionne aussi **hors-ligne** grâce à un repli synthétique automatique si le téléchargement des données échoue.

> 🛠️ **Données** : on tente de télécharger un jeu réel de résultats de matchs internationaux (1872–2024). Si ça échoue, un **générateur synthétique réaliste** prend le relais pour que le TP tourne **toujours**.

## 1. Imports et reproductibilité

On importe les outils habituels : `numpy`/`pandas` (données), `matplotlib` (graphiques), `tensorflow`/`keras` (le réseau), et quelques utilitaires `sklearn` (découpage train/test, normalisation, matrice de confusion).

On **fixe les graines aléatoires** (seed) pour que les résultats soient reproductibles : même graine → mêmes résultats.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# Reproductibilite : meme graine (seed) -> memes resultats
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow version :', tf.__version__)

## 2. Charger les données (réel + repli synthétique)

On tente de télécharger le CSV public **« International football results from 1872 »** (colonnes : `date`, `home_team`, `away_team`, `home_score`, `away_score`, `tournament`, `neutral`).

La fonction ci-dessous est **robuste** : si le téléchargement échoue (pas de réseau, URL indisponible…), un `try/except` bascule automatiquement sur un **générateur synthétique**. Dans tous les cas, elle imprime clairement la source utilisée.

> Pourquoi un repli ? Pour qu'un TP **tourne toujours**, même hors-ligne ou sur un Colab capricieux.

In [ ]:
URL_RESULTS = (
    'https://raw.githubusercontent.com/martj42/'
    'international-football-results-from-1872-to-2017/master/results.csv'
)


def generer_donnees_synthetiques(n_equipes=32, n_matchs=6000, seed=SEED):
    """Genere des matchs synthetiques realistes.

    Chaque equipe a une "force" cachee. Le resultat d'un match depend de la
    difference de force entre les deux equipes + l'avantage du terrain + du bruit.
    On renvoie un DataFrame au meme format que le CSV reel.
    """
    rng = np.random.default_rng(seed)
    equipes = ['Equipe_%02d' % i for i in range(n_equipes)]
    # Force cachee de chaque equipe (entre ~0 et ~3)
    force = {eq: rng.normal(1.5, 0.6) for eq in equipes}

    lignes = []
    for _ in range(n_matchs):
        a, b = rng.choice(equipes, size=2, replace=False)
        neutral = bool(rng.random() < 0.3)
        # Avantage du terrain pour l'equipe a domicile (sauf terrain neutre)
        avantage = 0.0 if neutral else 0.4
        lam_home = max(0.1, force[a] + avantage + rng.normal(0, 0.3))
        lam_away = max(0.1, force[b] + rng.normal(0, 0.3))
        # Nombre de buts ~ loi de Poisson (modele classique pour le foot)
        home_score = int(rng.poisson(lam_home))
        away_score = int(rng.poisson(lam_away))
        lignes.append({
            'date': '2024-01-01',
            'home_team': a,
            'away_team': b,
            'home_score': home_score,
            'away_score': away_score,
            'tournament': 'Synthetique',
            'neutral': neutral,
        })
    return pd.DataFrame(lignes)


def charger_donnees():
    """Tente de charger les donnees REELLES, sinon bascule sur le SYNTHETIQUE."""
    try:
        df = pd.read_csv(URL_RESULTS)
        # Verification basique des colonnes attendues
        cols = {'home_team', 'away_team', 'home_score', 'away_score', 'neutral'}
        if not cols.issubset(df.columns):
            raise ValueError('Colonnes inattendues dans le CSV.')
        df = df.dropna(subset=['home_score', 'away_score']).reset_index(drop=True)
        print('==> Donnees REELLES chargees :', len(df), 'matchs.')
        source = 'reel'
    except Exception as e:
        print('!! Telechargement impossible (', type(e).__name__, ') ->', e)
        df = generer_donnees_synthetiques()
        print('==> Repli SYNTHETIQUE utilise :', len(df), 'matchs.')
        source = 'synthetique'
    return df, source


df_brut, SOURCE = charger_donnees()
df_brut.head()

## 3. Construire les features + exploration rapide (EDA)

Un match brut (`home_score`, `away_score`) ne suffit pas : on veut prédire **avant** de connaître le score. On construit donc des **features** qui décrivent l'état des deux équipes **juste avant** le match, à partir de leur **historique récent** :

- **Forme récente** : moyenne de points sur les **5 derniers matchs** (victoire = 3 pts, nul = 1, défaite = 0).
- **Attaque / défense** : moyenne de buts **marqués** et **encaissés** sur ces 5 matchs.
- **Différences** entre domicile et extérieur (forme, attaque, défense).
- **Avantage du terrain** : 1 si l'équipe joue à domicile (terrain non neutre), 0 sinon.

La **cible** (ce qu'on prédit) est le résultat du match :
- `0` = victoire **extérieur**, `1` = **nul**, `2` = victoire **domicile**.

> C'est l'étape la plus importante d'un projet ML : un modèle ne vaut que par la **qualité des features** qu'on lui donne.

In [ ]:
def resultat_match(home_score, away_score):
    """0 = victoire exterieur, 1 = nul, 2 = victoire domicile."""
    if home_score > away_score:
        return 2
    if home_score < away_score:
        return 0
    return 1


def points(home_score, away_score, cote):
    """Points gagnes par une equipe ('home' ou 'away') sur un match."""
    r = resultat_match(home_score, away_score)
    if r == 1:
        return 1  # nul
    if cote == 'home':
        return 3 if r == 2 else 0
    return 3 if r == 0 else 0


def construire_features(df, fenetre=5):
    """Construit, pour chaque match, les features basees sur les 5 derniers
    matchs de chaque equipe (forme, buts marques/encaisses)."""
    df = df.reset_index(drop=True)
    # Historique par equipe : liste de (points, buts_marques, buts_encaisses)
    hist = {}

    def moyennes(equipe):
        h = hist.get(equipe, [])
        if len(h) == 0:
            return 1.0, 1.0, 1.0  # valeurs neutres au demarrage
        h = h[-fenetre:]
        pts = np.mean([x[0] for x in h])
        bm = np.mean([x[1] for x in h])
        be = np.mean([x[2] for x in h])
        return pts, bm, be

    lignes = []
    for _, row in df.iterrows():
        hs, as_ = int(row['home_score']), int(row['away_score'])
        neutral = bool(row['neutral'])

        forme_h, bm_h, be_h = moyennes(row['home_team'])
        forme_a, bm_a, be_a = moyennes(row['away_team'])

        lignes.append({
            'forme_home': forme_h,
            'forme_away': forme_a,
            'diff_forme': forme_h - forme_a,
            'buts_marques_home': bm_h,
            'buts_marques_away': bm_a,
            'buts_encaisses_home': be_h,
            'buts_encaisses_away': be_a,
            'diff_attaque': bm_h - bm_a,
            'diff_defense': be_a - be_h,
            'avantage_terrain': 0 if neutral else 1,
            'cible': resultat_match(hs, as_),
        })

        # Mise a jour de l'historique APRES avoir cree les features (pas de fuite)
        hist.setdefault(row['home_team'], []).append((points(hs, as_, 'home'), hs, as_))
        hist.setdefault(row['away_team'], []).append((points(hs, as_, 'away'), as_, hs))

    return pd.DataFrame(lignes)


df = construire_features(df_brut)
# On retire les premiers matchs ou l'historique est encore "neutre" pour les 2 equipes
print('Dataset de features :', df.shape)
df.head()

In [ ]:
# EDA : distribution des classes
noms_classes = ['Victoire exterieur (0)', 'Nul (1)', 'Victoire domicile (2)']
counts = df['cible'].value_counts().sort_index()

print('Repartition des resultats :')
for k, v in counts.items():
    print('  %-25s : %5d  (%.1f%%)' % (noms_classes[k], v, 100 * v / len(df)))

plt.figure(figsize=(6, 4))
plt.bar([noms_classes[i] for i in counts.index], counts.values,
        color=['#d62728', '#7f7f7f', '#2ca02c'])
plt.title('Distribution des classes (resultats de match)')
plt.ylabel('nombre de matchs')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 4. Prétraiter : séparation train/test + normalisation

Deux étapes classiques :
- **Découpage** train / test (80 / 20) avec `stratify` pour garder les mêmes proportions de classes des deux côtés.
- **Normalisation** avec `StandardScaler` : on met chaque feature à moyenne 0 et écart-type 1, ce qui aide l'optimisation à converger.

> ⚠️ Règle d'or **anti-fuite (data leakage)** : on **`fit` le scaler sur le train uniquement**, puis on **`transform`** train et test. Sinon, des informations du test « fuiteraient » dans l'entraînement.

In [ ]:
X = df.drop(columns=['cible']).values.astype('float32')
y = df['cible'].values.astype('int64')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# StandardScaler : fit sur le TRAIN uniquement, puis transform des deux cotes
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print('X_train :', X_train.shape, '| X_test :', X_test.shape)
print('Nombre de features par match :', X_train.shape[1])
print('Classes presentes :', np.unique(y_train))

## 5. Définir le MLP (réseau dense)

Notre réseau : `features → 64 → 32 → 3`.

- Chaque couche **`Dense`** est un ensemble de **neurones**. Un neurone calcule `somme(entrées × poids) + biais`, puis applique une **activation**.
- **ReLU** sur les couches cachées (garde les valeurs positives, met les négatives à 0).
- **Dropout** entre les couches : pendant l'entraînement, on « éteint » aléatoirement une fraction des neurones. C'est notre arme **anti-sur-apprentissage**.
- Couche de sortie : **3 neurones** avec **softmax** → 3 probabilités qui somment à 1 (une par classe).

Pour la **perte**, on utilise `sparse_categorical_crossentropy` car les labels sont des entiers (0, 1, 2).

In [ ]:
def construire_modele(n_features, dropout=0.3):
    modele = keras.Sequential([
        layers.Input(shape=(n_features,)),
        layers.Dense(64, activation='relu'),   # couche cachee 1 : 64 neurones
        layers.Dropout(dropout),               # anti-surapprentissage
        layers.Dense(32, activation='relu'),   # couche cachee 2 : 32 neurones
        layers.Dropout(dropout),
        layers.Dense(3, activation='softmax'),  # sortie : 3 classes (softmax)
    ])
    modele.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return modele


model = construire_modele(X_train.shape[1], dropout=0.3)
model.summary()

## 6. Entraîner et tracer les courbes

On entraîne sur plusieurs époques en réservant **20 % du train pour la validation**. Les courbes **train vs validation** nous disent si le modèle **sur-apprend** :

- Si la perte de **train** baisse mais que la perte de **validation remonte** → **sur-apprentissage** (le modèle mémorise au lieu de généraliser).
- Grâce au **dropout**, les deux courbes devraient rester proches.

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=40,
    batch_size=64,
    validation_split=0.2,
    verbose=0,
)
print('Entrainement termine sur', len(history.history['loss']), 'epoques.')

In [ ]:
def tracer_courbes(history):
    plt.figure(figsize=(11, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='train')
    plt.plot(history.history['val_loss'], label='validation')
    plt.title('Perte (loss)')
    plt.xlabel('epoque')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='train')
    plt.plot(history.history['val_accuracy'], label='validation')
    plt.title('Accuracy')
    plt.xlabel('epoque')
    plt.legend()

    plt.tight_layout()
    plt.show()


tracer_courbes(history)
print('Si les courbes train et validation restent proches : peu de surapprentissage.')
print('Si val_loss remonte alors que train_loss descend : surapprentissage.')

## 7. Évaluer sur le jeu de test + matrice de confusion

On mesure l'accuracy sur des matchs **jamais vus**, puis on lit la **matrice de confusion** : où le modèle se trompe-t-il ? (Souvent, le **nul** est la classe la plus difficile à prédire  c'est un résultat « entre les deux ».)

>  **Repère** : prédire le foot est dur. Une accuracy autour de **50–55 %** sur 3 classes est déjà honorable (le hasard donnerait ~33 %, et toujours prédire « victoire domicile » donne souvent ~45 %).

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print('Accuracy sur le test : %.4f  (perte : %.4f)' % (test_acc, test_loss))

# Baseline naive : toujours predire la classe majoritaire
classe_majoritaire = np.bincount(y_train).argmax()
baseline = (y_test == classe_majoritaire).mean()
print('Baseline (toujours predire la classe majoritaire) : %.4f' % baseline)

In [ ]:
y_pred = model.predict(X_test, verbose=0).argmax(axis=1)

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay(cm, display_labels=['ext (0)', 'nul (1)', 'dom (2)']).plot(
    ax=ax, cmap='Blues', colorbar=False
)
plt.title('Matrice de confusion (test)')
plt.show()

print(classification_report(y_test, y_pred,
                            target_names=['ext (0)', 'nul (1)', 'dom (2)'],
                            zero_division=0))

## 8. Bonus  mini-simulation de tournoi 🏆

Le modèle ne renvoie pas un simple « gagnant » : il renvoie **3 probabilités**. On peut donc simuler un match entre deux équipes en leur donnant des **statistiques fictives** (forme, attaque, défense) et lire les probabilités prédites.

La fonction `predire_match` ci-dessous construit un vecteur de features pour deux équipes, le **normalise avec le scaler entraîné**, et renvoie les probabilités.

In [ ]:
# Colonnes de features dans l'ordre exact attendu par le modele
COLS = list(df.drop(columns=['cible']).columns)


def predire_match(stats_home, stats_away, terrain_neutre=False, afficher=True):
    """Predit l'issue d'un match.

    stats_home / stats_away : dict avec les cles 'forme', 'buts_marques',
    'buts_encaisses' (valeurs typiques entre 0 et 3).
    """
    f = {
        'forme_home': stats_home['forme'],
        'forme_away': stats_away['forme'],
        'diff_forme': stats_home['forme'] - stats_away['forme'],
        'buts_marques_home': stats_home['buts_marques'],
        'buts_marques_away': stats_away['buts_marques'],
        'buts_encaisses_home': stats_home['buts_encaisses'],
        'buts_encaisses_away': stats_away['buts_encaisses'],
        'diff_attaque': stats_home['buts_marques'] - stats_away['buts_marques'],
        'diff_defense': stats_away['buts_encaisses'] - stats_home['buts_encaisses'],
        'avantage_terrain': 0 if terrain_neutre else 1,
    }
    x = np.array([[f[c] for c in COLS]], dtype='float32')
    x = scaler.transform(x)
    probs = model.predict(x, verbose=0)[0]
    if afficher:
        print('  P(victoire exterieur) = %.1f%%' % (100 * probs[0]))
        print('  P(match nul)          = %.1f%%' % (100 * probs[1]))
        print('  P(victoire domicile)  = %.1f%%' % (100 * probs[2]))
    return probs


# Exemple : une equipe domicile en grande forme contre une equipe plus faible
print('Match exemple : Bresil (domicile, forme) vs Outsider (exterieur)')
favori = {'forme': 2.6, 'buts_marques': 2.4, 'buts_encaisses': 0.7}
outsider = {'forme': 1.0, 'buts_marques': 1.0, 'buts_encaisses': 1.8}
_ = predire_match(favori, outsider, terrain_neutre=False)

In [ ]:
# Mini-tournoi : on confronte 4 equipes fictives, chacune avec ses stats
equipes = {
    'Bresil':    {'forme': 2.6, 'buts_marques': 2.4, 'buts_encaisses': 0.7},
    'France':    {'forme': 2.4, 'buts_marques': 2.1, 'buts_encaisses': 0.9},
    'Senegal':   {'forme': 1.9, 'buts_marques': 1.6, 'buts_encaisses': 1.1},
    'Outsider':  {'forme': 1.0, 'buts_marques': 1.0, 'buts_encaisses': 1.8},
}

issues = ['exterieur gagne', 'nul', 'domicile gagne']
noms = list(equipes.keys())
print('Demi-finales (terrain neutre) :\n')
for i in range(0, len(noms), 2):
    a, b = noms[i], noms[i + 1]
    probs = predire_match(equipes[a], equipes[b], terrain_neutre=True, afficher=False)
    pred = issues[probs.argmax()]
    print('%-9s vs %-9s -> %-16s (probas ext/nul/dom = %.2f / %.2f / %.2f)'
          % (a, b, pred, probs[0], probs[1], probs[2]))

## 9. À toi de jouer 🎮

Expérimente et observe l'effet sur l'**accuracy de test** et sur les **courbes** :

1. **Enlever le dropout** : appelle `construire_modele(X_train.shape[1], dropout=0.0)`, ré-entraîne, et regarde les courbes. La perte de **validation remonte-t-elle** pendant que celle du train continue de descendre ? C'est le **sur-apprentissage** en direct.
2. **Changer la taille des couches** : passe de `64 → 32` à `128 → 64`, ou ajoute une 3ème couche cachée. Plus gros = plus puissant, mais plus de risque de sur-apprendre.
3. **Plus / moins d'époques** : passe de 40 à 100. À partir de quand la validation cesse-t-elle de s'améliorer ?
4. **Ajouter des features** : dans `construire_features`, ajoute par exemple le **nombre de buts total** récent, ou agrandis la fenêtre (10 derniers matchs au lieu de 5).
5. **Régler le dropout** : teste `0.1`, `0.5`. Trop de dropout peut **sous-apprendre** (les deux courbes restent hautes).

> 📝 Note tes résultats dans un petit tableau : `config → accuracy test`. Quelle configuration généralise le mieux ?

In [ ]:
# Ton espace d'experimentation
# Exemple : un modele SANS dropout pour observer le surapprentissage
#
# model_overfit = construire_modele(X_train.shape[1], dropout=0.0)
# hist_overfit = model_overfit.fit(X_train, y_train, epochs=100,
#                                  batch_size=64, validation_split=0.2, verbose=0)
# tracer_courbes(hist_overfit)
# ...